In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import sys
import re

def parse_server_log(log_file):
    """Parse server log - formato correto"""
    data = {
        'time_sec': [],
        'rcvd_rate_mbps': [],
        'sent_rate_mbps': [],
        'rtt_ms': [],
        'mark_prob_percent': [],
        'pkt_mark': [],
        'loss_prob_percent': [],
        'pkt_lost': []
    }
    
    with open(log_file, 'r') as f:
        for line in f:
            if '[RECVER]:' not in line:
                continue
            
            try:
                match_ts = re.search(r'(\d+\.\d+)\s+sec', line)
                if not match_ts:
                    continue
                
                time_sec = float(match_ts.group(1))
                match_rates = re.search(r'Rcvd:\s+([\d.]+)\s+Mbps.*?Sent:\s+([\d.]+)\s+Mbps', line)
                if not match_rates:
                    continue
                
                rcvd = float(match_rates.group(1))
                sent = float(match_rates.group(2))
                
                match_rtt = re.search(r'RTT:\s+([\d.]+)\s+ms', line)
                rtt = float(match_rtt.group(1)) if match_rtt else 0.0
                
                match_mark = re.search(r'Mark:\s+([\d.]+)%\((\d+)/(\d+)\)', line)
                mark_prob = float(match_mark.group(1)) if match_mark else 0.0
                pkt_mark = int(match_mark.group(2)) if match_mark else 0
                
                match_loss = re.search(r'Lost:\s+([-\d.]+)%\((\d+)/(\d+)\)', line)
                loss_prob = float(match_loss.group(1)) if match_loss else 0.0
                pkt_lost = int(match_loss.group(2)) if match_loss else 0
                
                data['time_sec'].append(time_sec)
                data['rcvd_rate_mbps'].append(rcvd)
                data['sent_rate_mbps'].append(sent)
                data['rtt_ms'].append(rtt)
                data['mark_prob_percent'].append(mark_prob)
                data['pkt_mark'].append(pkt_mark)
                data['loss_prob_percent'].append(loss_prob)
                data['pkt_lost'].append(pkt_lost)
                
            except Exception as e:
                continue
    
    return pd.DataFrame(data)

def calculate_derived_metrics(df):
    """Calcular métricas derivadas a partir dos dados do servidor"""
    
    # 1. RTT Jitter (variância em janela móvel)
    window = 10
    df['rtt_jitter_ms'] = df['rtt_ms'].rolling(window=window, center=True).std()
    
    # 2. RTT Min/Max Ratio (em janela móvel)
    df['rtt_min_window'] = df['rtt_ms'].rolling(window=window, center=True).min()
    df['rtt_max_window'] = df['rtt_ms'].rolling(window=window, center=True).max()
    df['rtt_minmax_ratio'] = df['rtt_max_window'] / (df['rtt_min_window'] + 1e-6)
    
    # 3. Goodput (eficiência: dados úteis / overhead)
    # Assumindo ACKs são aproximadamente 10% do tráfego
    df['goodput'] = df['rcvd_rate_mbps'] / (df['sent_rate_mbps'] + 1e-6)
    
    # 4. Throughput Variance (estabilidade)
    df['rcvd_variance'] = df['rcvd_rate_mbps'].rolling(window=window, center=True).std()
    df['rcvd_cv'] = df['rcvd_variance'] / (df['rcvd_rate_mbps'] + 1e-6) * 100  # Coefficient of Variation %
    
    # 5. RTT Growth Rate (d(RTT)/dt) - derivada
    df['rtt_growth_rate'] = df['rtt_ms'].diff() / df['time_sec'].diff()
    
    # 6. Cumulative Metrics
    df['total_pkt_received'] = df['pkt_mark'].cumsum() + df['pkt_lost'].cumsum() + 1
    df['total_pkt_lost'] = df['pkt_lost'].cumsum()
    df['cumulative_loss_rate'] = (df['pkt_lost'].cumsum() / df['total_pkt_received']) * 100
    
    # 7. Buffer Depth Estimado (BDP - Bandwidth Delay Product)
    # BDP = bandwidth * RTT = (taxa em Mbps) * (RTT em s) / 8 (converter para bytes)
    df['buffer_depth_bytes'] = (df['rcvd_rate_mbps'] * df['rtt_ms'] / 1000) / 8 * 1e6
    
    # 8. ECN Efficiency (de todos os pacotes, quantos foram marcados?)
    df['ecn_efficiency'] = (df['pkt_mark'] / df['total_pkt_received']) * 100
    
    # 9. Retransmission Estimate (bytes perdidos/seg)
    # Se perder N% de throughput, N% é retransmissão
    df['retransmission_rate_kbps'] = df['rcvd_rate_mbps'] * df['loss_prob_percent'] / 100 * 1000
    
    # 10. Network Quality Index (0-100)
    # Combines RTT stability, loss rate, and ECN marking
    rtt_score = 100 - np.clip(df['rtt_jitter_ms'] / df['rtt_ms'].max() * 100, 0, 100)
    loss_score = 100 - np.clip(df['loss_prob_percent'] * 10, 0, 100)
    ecn_score = np.clip(df['mark_prob_percent'] * 2, 0, 100)  # Prefer some ECN marking
    
    df['network_quality_index'] = (rtt_score + loss_score + ecn_score) / 3
    
    return df

def create_enhanced_report(df, output_file='prague_performance_enhanced.png'):
    """Criar relatório com 8 gráficos incluindo métricas derivadas"""
    
    if df.empty:
        print("❌ Dataframe vazio")
        return
    
    fig = plt.figure(figsize=(22, 16))
    gs = fig.add_gridspec(4, 2, hspace=0.35, wspace=0.3)
    
    # ===== 1. RTT com Jitter =====
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(df['time_sec'], df['rtt_ms'], color='#2E86AB', linewidth=2.5, label='RTT', marker='o', markersize=2, markevery=15)
    ax1.fill_between(df['time_sec'], df['rtt_ms'], alpha=0.2, color='#2E86AB')
    
    # Envelope de jitter
    ax1_twin = ax1.twinx()
    ax1_twin.plot(df['time_sec'], df['rtt_jitter_ms'], color='#E74C3C', linewidth=1.5, linestyle='--', label='Jitter (σ)', alpha=0.7)
    ax1_twin.set_ylabel('Jitter (ms)', fontsize=10, color='#E74C3C', fontweight='bold')
    
    ax1.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax1.set_ylabel('RTT (ms)', fontsize=10, fontweight='bold')
    ax1.set_title('📊 RTT + Jitter Evolution', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_facecolor('#f8f9fa')
    ax1.legend(loc='upper left', fontsize=9)
    
    # ===== 2. RTT Min/Max Ratio (Estabilidade) =====
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.fill_between(df['time_sec'], 1, df['rtt_minmax_ratio'], alpha=0.4, color='#9B59B6', label='Max/Min Ratio')
    ax2.plot(df['time_sec'], df['rtt_minmax_ratio'], color='#8E44AD', linewidth=2.5, marker='d', markersize=3, markevery=15)
    ax2.axhline(y=1.2, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Estável (<1.2)')
    ax2.axhline(y=1.5, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Moderado (<1.5)')
    
    ax2.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax2.set_ylabel('RTT Max/Min Ratio', fontsize=10, fontweight='bold')
    ax2.set_title('📈 Estabilidade de RTT (Min/Max)', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_facecolor('#f8f9fa')
    ax2.legend(fontsize=9)
    
    # ===== 3. Goodput (Eficiência) =====
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(df['time_sec'], df['goodput'], color='#16A085', linewidth=2.5, label='Goodput Ratio', marker='s', markersize=3, markevery=15)
    ax3.axhline(y=1.0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Ideal (1.0)')
    ax3.fill_between(df['time_sec'], 0, df['goodput'], alpha=0.2, color='#16A085')
    
    ax3.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax3.set_ylabel('RX / TX Ratio', fontsize=10, fontweight='bold')
    ax3.set_title('💰 Goodput - Eficiência de Transferência', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.set_facecolor('#f8f9fa')
    ax3.legend(fontsize=9)
    
    # ===== 4. Throughput Coefficient of Variation =====
    ax4 = fig.add_subplot(gs[1, 1])
    colors_cv = ['#06A77D' if x < 5 else '#FFB703' if x < 10 else '#E74C3C' for x in df['rcvd_cv'].fillna(0)]
    ax4.bar(df['time_sec'], df['rcvd_cv'].fillna(0), width=0.6, color=colors_cv, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax4.axhline(y=5, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Estável (<5%)')
    ax4.axhline(y=10, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Moderado (<10%)')
    
    ax4.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax4.set_ylabel('CV (%)', fontsize=10, fontweight='bold')
    ax4.set_title('📉 Variância de Throughput RX', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.set_facecolor('#f8f9fa')
    ax4.legend(fontsize=9)
    
    # ===== 5. RTT Growth Rate (Congestionamento) =====
    ax5 = fig.add_subplot(gs[2, 0])
    rtt_growth = df['rtt_growth_rate'].fillna(0)
    colors_growth = ['#E74C3C' if x > 0 else '#06A77D' for x in rtt_growth]
    ax5.bar(df['time_sec'], rtt_growth, width=0.6, color=colors_growth, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax5.axhline(y=0, color='black', linestyle='-', linewidth=1)
    
    ax5.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax5.set_ylabel('d(RTT)/dt (ms/s)', fontsize=10, fontweight='bold')
    ax5.set_title('⚡ RTT Growth Rate (detecção congestionamento)', fontsize=12, fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='y')
    ax5.set_facecolor('#f8f9fa')
    
    # ===== 6. Cumulative Loss Rate =====
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.plot(df['time_sec'], df['cumulative_loss_rate'], color='#E74C3C', linewidth=2.5, label='Cumulative Loss %', marker='o', markersize=2, markevery=15)
    ax6.fill_between(df['time_sec'], df['cumulative_loss_rate'], alpha=0.2, color='#E74C3C')
    ax6.axhline(y=1.0, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='1% Threshold')
    
    ax6.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax6.set_ylabel('Cumulative Loss Rate (%)', fontsize=10, fontweight='bold')
    ax6.set_title('📊 Taxa de Perda Acumulada', fontsize=12, fontweight='bold')
    ax6.grid(True, alpha=0.3)
    ax6.set_facecolor('#f8f9fa')
    ax6.legend(fontsize=9)
    
    # ===== 7. Buffer Depth (BDP) =====
    ax7 = fig.add_subplot(gs[3, 0])
    buffer_mb = df['buffer_depth_bytes'] / (1024 * 1024)
    ax7.fill_between(df['time_sec'], buffer_mb, alpha=0.4, color='#3498DB', label='Estimated Buffer Depth')
    ax7.plot(df['time_sec'], buffer_mb, color='#2980B9', linewidth=2.5, marker='s', markersize=3, markevery=15)
    
    ax7.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax7.set_ylabel('Buffer Depth (MB)', fontsize=10, fontweight='bold')
    ax7.set_title('💾 Estimated Network Buffer Depth (BDP)', fontsize=12, fontweight='bold')
    ax7.grid(True, alpha=0.3)
    ax7.set_facecolor('#f8f9fa')
    
    # ===== 8. Network Quality Index =====
    ax8 = fig.add_subplot(gs[3, 1])
    nqi = df['network_quality_index'].fillna(0)
    colors_nqi = ['#E74C3C' if x < 50 else '#FFB703' if x < 70 else '#06A77D' for x in nqi]
    ax8.bar(df['time_sec'], nqi, width=0.6, color=colors_nqi, alpha=0.8, edgecolor='black', linewidth=0.5, label='NQI')
    ax8.axhline(y=70, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Boa (>70)')
    ax8.axhline(y=50, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Moderada (>50)')
    
    ax8.set_xlabel('Tempo (s)', fontsize=10, fontweight='bold')
    ax8.set_ylabel('Network Quality Index (0-100)', fontsize=10, fontweight='bold')
    ax8.set_title('🎯 Índice de Qualidade de Rede', fontsize=12, fontweight='bold')
    ax8.set_ylim(0, 100)
    ax8.grid(True, alpha=0.3, axis='y')
    ax8.set_facecolor('#f8f9fa')
    ax8.legend(fontsize=9)
    
    fig.suptitle('🚀 Análise Avançada de Performance - Prague Congestion Control\nMétricas Derivadas apenas do Servidor',
                 fontsize=16, fontweight='bold', y=0.995)
    
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✅ Relatório avançado salvo em: {output_file}")
    plt.show()

def create_advanced_statistics(df, output_csv='prague_advanced_metrics.csv'):
    """Exportar métricas avançadas para CSV"""
    
    stats = {
        'Métrica': [
            '--- BASIC METRICS ---',
            'Duração do Teste (s)',
            'RTT Médio (ms)',
            'RTT Min (ms)',
            'RTT Max (ms)',
            'RTT Desvio Padrão (ms)',
            '',
            '--- DERIVED METRICS ---',
            'Jitter Médio (σ RTT)',
            'Jitter Máximo',
            'RTT Max/Min Ratio Médio',
            'Estabilidade RTT (%)',
            '',
            '--- EFFICIENCY ---',
            'Goodput Médio',
            'Goodput Min',
            'Goodput Max',
            'Variance CV (%)',
            'Throughput RX Médio (Mbps)',
            '',
            '--- LOSS & ECN ---',
            'Taxa de Perda Média (%)',
            'Taxa de Perda Máxima (%)',
            'Total Pacotes Perdidos',
            'Taxa ECN Média (%)',
            'Taxa ECN Máxima (%)',
            'Total Pacotes Marcados ECN',
            '',
            '--- NETWORK HEALTH ---',
            'Buffer Depth Médio (MB)',
            'Buffer Depth Máximo (MB)',
            'RTT Growth Rate Médio (ms/s)',
            'Retransmission Rate Média (Kbps)',
            'Network Quality Index Médio',
            'Intervalos com NQI Bom (>70)',
            'Intervalos com NQI Moderado (50-70)',
            'Intervalos com NQI Ruim (<50)',
        ],
        'Valor': [
            '',
            f"{df['time_sec'].max():.2f}",
            f"{df['rtt_ms'].mean():.2f}",
            f"{df['rtt_ms'].min():.2f}",
            f"{df['rtt_ms'].max():.2f}",
            f"{df['rtt_ms'].std():.2f}",
            '',
            '',
            f"{df['rtt_jitter_ms'].mean():.2f}",
            f"{df['rtt_jitter_ms'].max():.2f}",
            f"{df['rtt_minmax_ratio'].mean():.2f}",
            f"{(100 - (df['rtt_jitter_ms'].mean() / df['rtt_ms'].max() * 100)):.1f}",
            '',
            '',
            f"{df['goodput'].mean():.4f}",
            f"{df['goodput'].min():.4f}",
            f"{df['goodput'].max():.4f}",
            f"{df['rcvd_cv'].mean():.2f}",
            f"{df['rcvd_rate_mbps'].mean():.4f}",
            '',
            '',
            f"{df['loss_prob_percent'].mean():.3f}",
            f"{df['loss_prob_percent'].max():.3f}",
            f"{df['pkt_lost'].sum():.0f}",
            f"{df['mark_prob_percent'].mean():.3f}",
            f"{df['mark_prob_percent'].max():.3f}",
            f"{df['pkt_mark'].sum():.0f}",
            '',
            '',
            f"{(df['buffer_depth_bytes'].mean() / (1024*1024)):.2f}",
            f"{(df['buffer_depth_bytes'].max() / (1024*1024)):.2f}",
            f"{df['rtt_growth_rate'].mean():.3f}",
            f"{df['retransmission_rate_kbps'].mean():.2f}",
            f"{df['network_quality_index'].mean():.1f}",
            f"{(df['network_quality_index'] > 70).sum()}",
            f"{((df['network_quality_index'] >= 50) & (df['network_quality_index'] <= 70)).sum()}",
            f"{(df['network_quality_index'] < 50).sum()}",
        ]
    }
    
    stats_df = pd.DataFrame(stats)
    stats_df.to_csv(output_csv, index=False)
    print(f"✅ Métricas avançadas salvas em: {output_csv}")
    
    print("\n" + "="*70)
    print("📊 RESUMO ESTATÍSTICO COMPLETO")
    print("="*70)
    for idx, row in stats_df.iterrows():
        if row['Métrica'] == '':
            print()
        elif row['Métrica'].startswith('---'):
            print(f"\n{row['Métrica']}")
        else:
            print(f"{row['Métrica']:.<50} {row['Valor']:>17}")
    print("="*70 + "\n")

def main():
    if len(sys.argv) < 2:
        print("Uso: python generate_performance_graphs_v2.py <arquivo_log> [imagem] [csv]")
        print("Exemplo: python generate_performance_graphs_v2.py server.log report.png metrics.csv")
        sys.exit(1)
    
    log_file = "server.log"
    output_image = sys.argv[2] if len(sys.argv) > 2 else 'prague_performance_enhanced.png'
    output_csv = sys.argv[3] if len(sys.argv) > 3 else 'prague_advanced_metrics.csv'
    
    print(f"\n📂 Lendo log: {log_file}")
    df = parse_server_log(log_file)
    
    if df.empty:
        print("❌ Nenhum dado encontrado")
        sys.exit(1)
    
    print(f"✅ {len(df)} pontos de dados parseados")
    
    print(f"\n🔧 Calculando {10} métricas derivadas...")
    df = calculate_derived_metrics(df)
    
    print(f"\n📊 Gerando relatório visual...")
    create_enhanced_report(df, output_image)
    
    print(f"\n📋 Exportando estatísticas...")
    create_advanced_statistics(df, output_csv)
    
    print(f"\n✨ Concluído! Arquivos:")
    print(f"   📊 {output_image}")
    print(f"   📋 {output_csv}")

if __name__ == '__main__':
    main()